# ch08 — 임계값: EVT(SPOT)와 conformal

In [ ]:
import numpy as np


In [ ]:
from tsad_forge.evaluation.thresholding import (
    conformal_threshold,
    quantile_threshold,
    spot_threshold,
)

rng = np.random.default_rng(0)
cal = rng.normal(size=5000)                       # 정상(보정) 점수
test_scores = np.concatenate([rng.normal(size=2000), rng.normal(5, 1, size=20)])  # 이상 20개

for name, th in [
    ("quantile(0.99)", quantile_threshold(cal, 0.99)),
    ("SPOT(q=1e-3)", spot_threshold(test_scores, q=1e-3, calibration=cal)),
    ("conformal(a=0.01)", conformal_threshold(test_scores, alpha=0.01, calibration=cal)),
]:
    pred = test_scores >= th
    fp = pred[:2000].mean(); tp = pred[2000:].mean()
    print(f"{name:18s} th={th:6.3f}  오탐률={fp:.4f}  이상탐지율={tp:.2f}")

In [ ]:
# best-F1(oracle)이 배포에서 재현 불가능한 이유: test 라벨을 몰래 본 값이다
from tsad_forge.evaluation.metrics import compute_metrics

labels = np.concatenate([np.zeros(2000, dtype=int), np.ones(20, dtype=int)])
m = compute_metrics(test_scores, labels)
print(f"best_f1(oracle)={m['best_f1']:.3f} — 리더보드 참고용일 뿐, 운영 임계값이 아니다")